# Week 49, Day 4: Language models and the brain: encoding models and brain scores

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w049/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Cross-validated ridge encoding

Write cv_score(F, Y): 5-fold cross-validation; in each fold fit ridge weights W = (F^T F + lam I)^-1 F^T Y on the training sentences (lam = 10, features and responses centred on the training set), predict the test sentences, and finally return each voxel's correlation between all held-out predictions and responses.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2021)
n_sent, n_lat, n_vox = 200, 20, 50
Z = r.normal(size=(n_sent, n_lat))                       # latent sentence properties
B = r.normal(size=(n_lat, n_vox))
signal = Z @ B / np.sqrt(n_lat)
Y1 = signal + 1.0 * r.normal(size=signal.shape)          # repetition 1
Y2 = signal + 1.0 * r.normal(size=signal.shape)          # repetition 2
Y = (Y1 + Y2) / 2
F_info = Z @ r.normal(size=(n_lat, 100)) + 0.5 * r.normal(size=(n_sent, 100))
F_rand = r.normal(size=(n_sent, 100))

def cv_score(F, Y, lam=10.0, k=5):
    folds = np.array_split(r.permutation(len(F)), k); pred = np.zeros_like(Y)
    for te in folds:
        tr = np.setdiff1d(np.arange(len(F)), te)
        ...
    return np.array([np.corrcoef(pred[:, v], Y[:, v])[0, 1] for v in range(Y.shape[1])])

score = {"informative": cv_score(F_info, Y), "random": cv_score(F_rand, Y)}
for k_, s in score.items():
    print(f"{k_:11s}: median held-out r = {np.median(s):.3f}")

In [ ]:
import numpy as np
def test_good():
    "the informative model's median held-out correlation exceeds 0.4"
    assert np.median(score["informative"]) > 0.4
def test_random():
    "the random model's median held-out correlation is below 0.1"
    assert abs(np.median(score["random"])) < 0.1

test_good()
test_random()
print("All checks passed")

### Exercise: How good could any model be?

For each voxel compute r12 = corr(Y1, Y2), the Spearman-Brown reliability of the mean, and the ceiling as its square root. Divide the informative model's scores by the ceiling.

In [ ]:
r12 = np.array([np.corrcoef(Y1[:, v], Y2[:, v])[0, 1] for v in range(n_vox)])
ceiling = ...
norm_pred = score["informative"] / ceiling
print(f"median split-half r {np.median(r12):.3f}; median ceiling {np.median(ceiling):.3f}; median normalised predictivity {np.median(norm_pred):.3f}")
plt.scatter(ceiling, score["informative"]); plt.plot([0, 1], [0, 1], "k:"); plt.xlabel("noise ceiling"); plt.ylabel("model r"); plt.show()

In [ ]:
import numpy as np
def test_ceiling():
    "the median ceiling is between 0.6 and 0.95"
    assert 0.6 < np.median(ceiling) < 0.95
def test_normalised():
    "the informative model's median normalised predictivity is above 0.6"
    assert np.median(norm_pred) > 0.6

test_ceiling()
test_normalised()
print("All checks passed")

### Exercise: Brain score against task performance

For q from 0 to 1 in 8 steps, build features q Z M + (1 - q) noise, compute the median cross-validated brain score and a task score (cross-validated correlation for predicting the first latent property), and correlate the two across models.

In [ ]:
qs = np.linspace(0, 1, 8); brain, task = [], []
for q in qs:
    F = q * (Z @ r.normal(size=(n_lat, 100))) + (1 - q) * 3 * r.normal(size=(n_sent, 100))
    brain.append(...)
    task.append(...)
brain, task = np.array(brain), np.array(task)
print("brain scores:", np.round(brain, 2)); print("task scores: ", np.round(task, 2))
print(f"correlation across models {np.corrcoef(brain, task)[0, 1]:.2f}")
plt.plot(task, brain, "o"); plt.xlabel("task (prediction) score"); plt.ylabel("brain score"); plt.show()

In [ ]:
import numpy as np
def test_corr():
    "across models, brain score and task score correlate above 0.8"
    assert np.corrcoef(brain, task)[0, 1] > 0.8

test_corr()
print("All checks passed")